In [1]:
import pandas as pd
import numpy as np
print(pd.__version__)
print(np.__version__)

3.0.0
2.4.2


In [2]:
import os
import json
import pickle
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm.notebook import tqdm

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [3]:
files = sorted(Path("processed_chunks_paquet").glob("chunk_*.parquet"))


for file in files[:3]:
    df = pd.read_parquet(file)
    
    print(file.name)
    print(df.head())

chunk_000.parquet
   Unnamed: 0         id               domain        type  \
0         732  7444726.0   nationalreview.com   political   
1        1348  6213642.0    beforeitsnews.com        fake   
2        7119  3867639.0     dailycurrant.com      satire   
3        1518  9560791.0          nytimes.com    reliable   
4        9345  2059625.0  infiniteunknown.net  conspiracy   

                                                 url  \
0  http://www.nationalreview.com/node/152734/%E2%...   
1  http://beforeitsnews.com/economy/2012/06/the-c...   
2  http://dailycurrant.com/2016/01/18/man-awoken-...   
3  https://query.nytimes.com/gst/fullpage.html?re...   
4  http://www.infiniteunknown.net/2011/09/14/100-...   

                   scraped_at                 inserted_at  \
0  2017-11-27T01:14:42.983556  2018-02-08 19:18:34.468038   
1    2017-11-27T01:14:08.7454  2018-02-08 19:18:34.468038   
2  2017-11-27T01:14:21.395055  2018-02-07 23:39:33.852671   
3  2018-02-11 00:46:42.632962  201

In [4]:
from sklearn.model_selection import train_test_split

fake_categories = {"fake", "hate", "unreliable", "clickbait", "conspiracy", "junksci", "rumor"}
valid_types = fake_categories | {"reliable", "political", "bias"}

def load_chunks_logistic(files):
    dfs = []
    for file in tqdm(files):
        df = pd.read_parquet(file)
        df = df[df["type"].isin(valid_types)][["type", "tokens_stemmed", "domain"]]
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)
    del dfs
    data["text"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))
    data["text_and_domain"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens)) + " " + data["domain"]
    data["label"] = data["type"].apply(lambda x: 1 if x in fake_categories else 0)
    return data[["text", "text_and_domain", "label"]]

all_data = load_chunks_logistic(files[:100])

# First split off 20% for val+test
train_data, temp_data = train_test_split(all_data, test_size=0.2, random_state=42)

# Split the 20% evenly into val and test
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

X_train, y_train = train_data["text"], train_data["label"]
X_val,   y_val   = val_data["text"],   val_data["label"]
X_test,  y_test  = test_data["text"],  test_data["label"]

X_tr_domain = train_data["text_and_domain"]
X_v_domain  = val_data["text_and_domain"]
X_te_domain = test_data["text_and_domain"]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")



  0%|          | 0/100 [00:00<?, ?it/s]

Train: 712000, Val: 89000, Test: 89001


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

#Vi bruger TfidVectorizer og ComplementNB, som er tools til at bygge en pipeline dataen. TF-IDF omdanner ord til vægte baseret på # hvor sjældne de er på tværs af artikler, og ComplementNB klassificerer ud fra de vægte.
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, sublinear_tf=True)),
    ("clf",   ComplementNB())
])

#Vi bruger et range af alpha-værdier til Laplace smoothing. Alpha styrer hvor meget vi udjævner sandsynligheder for sjældne ord - dette er så vi ikke lader et ukendt ord ødelægge hele prediktionen
alphas = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
best_alpha, best_f1 = None, 0

for alpha in alphas:
    pipeline.set_params(clf__alpha=alpha)
    pipeline.fit(X_train, y_train)
    f1 = f1_score(y_val, pipeline.predict(X_val))
    print(f"alpha={alpha:.2f}  val F1={f1:.4f}")
    # Her gemmer vi alpha hvis dens F1-værdi på validation settet er bedre end den hidtil bedste - på den måde ender vi med den alpha der generaliserer bedst. 
    if f1 > best_f1:
        best_f1, best_alpha = f1, alpha

#Træn med bedste alpha og evaluer på test
pipeline.set_params(clf__alpha=best_alpha)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["reliable", "fake"]))

alpha=0.01  val F1=0.7879
alpha=0.10  val F1=0.7872
alpha=0.50  val F1=0.7861
alpha=1.00  val F1=0.7857
alpha=5.00  val F1=0.7863
alpha=10.00  val F1=0.7835
              precision    recall  f1-score   support

    reliable       0.90      0.80      0.85     54574
        fake       0.73      0.86      0.79     34427

    accuracy                           0.82     89001
   macro avg       0.82      0.83      0.82     89001
weighted avg       0.84      0.82      0.82     89001



In [6]:
#preprocessing for liar

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize_text(text):
    text = str(text).lower()
    return word_tokenize(text)

def clean_tokens(tokens):
    return [t for t in tokens if t.isalpha() and t not in stop_words]

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

In [10]:
## Liar
import pandas as pd
# Use the split you want to evaluate on
liar_raw = pd.read_csv("liar_dataset/test.tsv", sep="\t", header=None)
print(liar_raw.shape)
print(liar_raw.head(2))
# LIAR has no header row, so we assign column names
liar_raw.columns = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "speaker_job_title",
    "state_info",
    "party_affiliation",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_fire_counts",
    "context"
]

""" print(liar_raw.head())
print(liar_raw["label"].value_counts()) """
#Samme kategorier tværs modellerne for kontinuitet og mere målbare resultater 
fake_categories_liar = {"false", "pants-fire", "barely-true"}
valid_types_liar = fake_categories_liar | {"true", "mostly-true"}

# Group mapping
fake_categories_liar = {"false", "pants-fire", "barely-true"}
real_categories_liar = {"true", "mostly-true"}
valid_categories_liar = fake_categories_liar | real_categories_liar

# Find tekstkolonnen
TEXT_COL = "text"
if TEXT_COL not in liar_raw.columns:
    TEXT_COL = "statement"

# Lav clean liar_df
liar_df = liar_raw.copy()

# Behold kun de labels vi vil bruge
liar_df = liar_df[liar_df["label"].isin(valid_categories_liar)].copy()

# Fjern manglende/blank tekst
liar_df = liar_df[liar_df[TEXT_COL].notna()].copy()
liar_df[TEXT_COL] = liar_df[TEXT_COL].astype(str).str.strip()
liar_df = liar_df[liar_df[TEXT_COL] != ""].copy()

# Preprocess LIAR text
liar_df["tokens"] = liar_df[TEXT_COL].apply(tokenize_text)
liar_df["tokens_clean"] = liar_df["tokens"].apply(clean_tokens)
liar_df["tokens_stemmed"] = liar_df["tokens_clean"].apply(stem_tokens)
liar_df["text"] = liar_df["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))

# Map til binær target
# 0 = fake, 1 = real
liar_df["target"] = liar_df["label"].apply(
    lambda x: 0 if x in fake_categories_liar else 1
)

# LIAR (cross-domain, no retraining)
y_pred_liar = pipeline.predict(liar_df["text"])
print("LIAR Dataset (cross-domain)")
print(classification_report(liar_df["target"], y_pred_liar, target_names=["real", "fake"]))

#Lasse's idé.
avg_tokens = train_data["text"].apply(lambda x: len(x.split())).mean()
print(f"Average token length: {avg_tokens:.1f}")




(1267, 14)
           0      1                                                  2   \
0  11972.json   true  Building a wall on the U.S.-Mexico border will...   
1  11685.json  false  Wisconsin is on pace to double the number of l...   

            3                  4                     5          6   \
0  immigration         rick-perry              Governor      Texas   
1         jobs  katrina-shankland  State representative  Wisconsin   

           7   8   9   10  11  12                 13  
0  republican  30  30  42  23  18    Radio interview  
1    democrat   2   1   0   0   0  a news conference  
LIAR Dataset (cross-domain)
              precision    recall  f1-score   support

        real       0.54      0.71      0.61       553
        fake       0.43      0.27      0.33       449

    accuracy                           0.51      1002
   macro avg       0.48      0.49      0.47      1002
weighted avg       0.49      0.51      0.49      1002

Average token length: 251.9
